# 현실의 데이터 02: 쇼핑몰 구매 전환 예측

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/datashop/02_shop_conversion.ipynb)

**분야**: 커머스 · 콘텐츠 &nbsp;|&nbsp; **이 분야에서 먼저 보는 문제**: 브랜드 · 콘텐츠 마케팅 · 직접판매 전환율 최적화 · 해외직구 커머스

> 쇼핑몰 방문 1만 2천 건. 이 방문자는 살까, 그냥 나갈까?

**데이터**: Online Shoppers Purchasing Intention (약 1MB). UCI (Sakar et al., 2019), 온라인 쇼핑몰 방문 세션 12,330건 (1년, 사용자당 1세션).

인턴 · 산학협력 과제에서 처음 받게 될 데이터와 닮은 공개 데이터다. 데이터를 열어 보고, 아래 예시처럼 문제 하나를 정해 기준선과 비교한 뒤, 맨 끝의 **나의 다섯 칸** 을 채운다 (1주 2차시 '문제 정형화와 평가지표').

| 칸 | 이 노트북의 예시 |
|---|---|
| ① 한 행 | 방문 세션 하나 |
| ② 입력 X | 본 페이지 수 · 머문 시간 · 이탈률 · 월 · 방문자 유형 … |
| ③ 출력 y | 구매했나 (Revenue) |
| ④ 평가지표 | 정밀도 · 재현율 |
| ⑤ 기준선 | 아무도 안 산다 |

**논문 · 출처** (직접 열어 보자: 이 데이터를 누가, 왜, 어떻게 만들었나):
- [Sakar et al. (2019) Real-time prediction of online shoppers' purchasing intention using MLP and LSTM, Neural Computing and Applications](https://doi.org/10.1007/s00521-018-3523-0)

In [1]:
import io, os, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url, path):
    """Download url to path once (skip if it already exists)"""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    if not os.path.exists(path):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (ML course data shop)"})
        with urllib.request.urlopen(req, timeout=600) as r, open(path, "wb") as f:
            f.write(r.read())
    return path

## 입장: 데이터 받기

In [2]:
URL = "https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip"
with zipfile.ZipFile(download(URL, "data/shoppers.zip")) as z:
    df = pd.read_csv(z.open("online_shoppers_intention.csv"))
print(df.shape)
df.head()

(12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


## 열어 보기

구매한 방문은 몇 %? 달마다 다를까?

In [3]:
print(f"conversion rate: {df['Revenue'].mean():.1%}")
by_month = df.groupby("Month")["Revenue"].agg(["mean", "size"]).sort_values("size", ascending=False)
print(by_month.round(3))
print(df.groupby("VisitorType")["Revenue"].mean().round(3))

conversion rate: 15.5%
        mean  size
Month             
May    0.109  3364
Nov    0.254  2998
Mar    0.101  1907
Dec    0.125  1727
Oct    0.209   549
Sep    0.192   448
Aug    0.176   433
Jul    0.153   432
June   0.101   288
Feb    0.016   184
VisitorType
New_Visitor          0.249
Other                0.188
Returning_Visitor    0.139
Name: Revenue, dtype: float64


## 문제 하나 골라 보기: 방문 → 살까?

기준선 = 아무도 안 산다. 모델은 그레이디언트 부스팅 (7주차). 그리고 **`PageValues` 를 넣을 때와 뺄 때** 를 비교한다.
`PageValues` 는 '이 방문자가 본 페이지들이 과거 거래에서 얼마짜리였나' 를 구글 애널리틱스가 계산한 값이다. 방문이 **끝나기 전에** 알 수 있는 값일까?

In [4]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

def precision_recall(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1)); fp = np.sum((y_pred == 1) & (y_true == 0)); fn = np.sum((y_pred == 0) & (y_true == 1))
    return (tp / (tp + fp) if tp + fp else 0.0), tp / (tp + fn)

X = pd.get_dummies(df.drop(columns="Revenue"), columns=["Month", "VisitorType"]).astype(float)
y = df["Revenue"].astype(int).values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

p, r = precision_recall(yte, np.zeros(len(yte), dtype=int))
print(f"{'nobody buys':26s} accuracy {np.mean(yte == 0):.1%}  precision {p:.0%}  recall {r:.0%}")
for name, cols in [("with PageValues", X.columns), ("without PageValues", [c for c in X.columns if c != "PageValues"])]:
    m = HistGradientBoostingClassifier(random_state=0).fit(Xtr[cols], ytr)
    pred = m.predict(Xte[cols]); p, r = precision_recall(yte, pred)
    print(f"{name:26s} accuracy {np.mean(yte == pred):.1%}  precision {p:.0%}  recall {r:.0%}")

nobody buys                accuracy 84.5%  precision 0%  recall 0%


with PageValues            accuracy 90.1%  precision 73%  recall 57%


without PageValues         accuracy 84.2%  precision 46%  recall 12%


**관찰**: `PageValues` 하나가 성능을 크게 바꾼다. 방문 도중에 '살지' 를 맞혀 쿠폰을 띄우는 게 목적이라면, 그 순간에 이 값을 알 수 있는지부터 확인해야 한다.

## 다른 문제로 바꿔 보기

- **전환율 최적화**: '살까?' 대신 '쿠폰을 보여 주면 살 사람' 을 찾고 싶다면 y 가 달라진다 (쿠폰을 보여 준 실험 데이터가 필요).
- **콘텐츠 마케팅**: 한 행을 '방문' 이 아니라 '광고 · 콘텐츠 하나' 로 정하면?
- **해외직구**: 리뷰 데이터 (예: Amazon Reviews 2023 의 Beauty 분야, McAuley Lab) 로 '재구매할 상품' 을 맞힐 수도 있다.

### 나의 다섯 칸

아래 표를 직접 채워 보자. 위의 예시와 다르게 정해도 된다.

| 칸 | 내가 정한 것 |
|---|---|
| ① 한 행은 무엇? | |
| ② 입력 X (예측할 때 알 수 있는 것만) | |
| ③ 출력 y | |
| ④ 평가지표 | |
| ⑤ 기준선 | |